# core

> Using pyinstrument to profile FastHTML apps.

Sometimes when building FastHTML apps we run into performance bottlenecks. Figuring out what is slow can be challenging, especially when building apps with async components. That's where profiling tools like pyinstrument can help. Profilers are tools that show exactly how long each component of a project takes to run. Identifying slow parts of an app is the first step in figuring out how to make things run faster.

In [ ]:
#| default_exp core

In [ ]:
#| export
from fasthtml.common import *
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.middleware import Middleware
try: from rich import print
except: pass
try:
    from pyinstrument import Profiler
except ImportError:
    raise ImportError('Please install pyinstrument')

In [ ]:
#| export
class ProfileMiddleware(BaseHTTPMiddleware):    
    async def dispatch(self, request, call_next):
        profiling = request.query_params.get("profile", False)
        term = request.query_params.get("term", False)  
        if profiling:
            profiler = Profiler()
            profiler.start()
            response = await call_next(request)            
            profiler.stop()
            if term: 
                print(profiler.output_text())
                return response
            return HTMLResponse(profiler.output_html())
        return await call_next(request)

## Tests

In [ ]:
from starlette.testclient import TestClient
from fastcore.all import *
from httpx import ASGITransport, AsyncClient
from functools import partialmethod
from anyio import from_thread

In [ ]:
class Client:
    "A simple httpx ASGI client that doesn't require `async`"
    def __init__(self, app, url="http://testserver"):
        self.cli = AsyncClient(transport=ASGITransport(app), base_url=url)

    def _sync(self, method, url, **kwargs):
        async def _request(): return await self.cli.request(method, url, **kwargs)
        with from_thread.start_blocking_portal() as portal: return portal.call(_request)

for o in ('get', 'post', 'delete', 'put', 'patch', 'options'): setattr(Client, o, partialmethod(Client._sync, o))

In [ ]:
app, rt = fast_app()
app.add_middleware(ProfileMiddleware)
client = Client(app)

First, confirm that the view works normally

In [ ]:
@rt
def index(): return Titled('Hello, profiler')
'Hello, profiler' in client.get('/').text

True

Now lets profile it! Or rather, check that it works.

In [ ]:
'pyinstrumentHTMLRenderer' in client.get('/?profile=1').text

True

Let's print to the terminal

In [ ]:
client.get('/?profile=1&term=1')

_     ._   __/__   _ _  _  _ _/_   Recorded: 15:19:47  Samples:  1
 /_//_/// /_\ / //_// / //_'/ //     Duration: 0.001     CPU time: 0.001
/   _/                      v5.0.1

Profile at /var/folders/3k/p8fttyg52s30bj9vdr1v7c180000gn/T/ipykernel_18206/207799178.py:8

0.001 ProfileMiddleware.dispatch  /var/folders/3k/p8fttyg52s30bj9vdr1v7c180000gn/T/ipykernel_18206/207799178.py:3
`- 0.001 Profiler.stop  pyinstrument/profiler.py:168
   `- 0.001 StackSampler.unsubscribe  pyinstrument/stack_sampler.py:96

<Response [200 OK]>

In [ ]:
%%ai
Write a FastHTML route decorator called 'instrument' that when added to a route adds `profile: 1` to `request.query_params` so the ProfileMiddleware activates

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()